# Wax strip cell counting
Scan PDF -> strips are cut automatically -> filled cells are segmented (Cellpose) -> counts per strip.

Runtime -> Change runtime type -> **T4 GPU**. Then: click **Upload PDF(s)**, wait until the file name appears, and run the second cell.

## 1. Upload PDF(s)
Click the button and choose your PDF(s). Wait until the file name shows next to it before running the next cell.

In [ ]:
from google.colab import files
import os

# Prepare input directory
os.makedirs('/content/input_pdfs', exist_ok=True)
print("Click 'Choose Files' below to upload your PDF(s):")

uploaded = files.upload()
pdf_paths = []
for name, content in uploaded.items():
    if name.lower().endswith('.pdf'):
        save_path = os.path.join('/content/input_pdfs', name)
        with open(save_path, 'wb') as f:
            f.write(content)
        pdf_paths.append(save_path)
        print(f"Saved: {name}")

if not pdf_paths:
    print("Warning: No PDF files were uploaded.")

FileUpload(value={}, accept='.pdf', description='Upload PDF(s)', layout=Layout(width='220px'), multiple=True)

## 2. Run everything
Installs, loads the model from Drive (`MyDrive/wax_cpsam`), cuts strips, segments, counts and downloads a results zip.

In [2]:
import os, sys, subprocess
!pip install -q cellpose==4.2.1.1 opencv-python-headless tifffile pymupdf
os.chdir('/content')
if not os.path.exists('recapping_Analysis'):
    subprocess.run(['git', 'clone', 'https://github.com/Mateo-Schaerer-Gonzalez/recapping_Analysis.git'], check=True)
else:
    subprocess.run(['git', '-C', 'recapping_Analysis', 'pull'], check=True)
os.chdir('/content/recapping_Analysis'); sys.path.insert(0, os.getcwd())
import wax_pipeline as wp
pdfs = wp.collect_pdfs(uploader)
counts = wp.colab_main(pdfs)
counts